In [ ]:
import os
import json
import re
import string
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models, callbacks, losses

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
!wget -q https://raw.githubusercontent.com/Cyral/Bakeoof/master/full_format_recipes.json \
    -O /content/full_format_recipes.json

In [ ]:
!ls -lh /content/full_format_recipes.json

In [ ]:
import json

DATA_PATH = "/content/full_format_recipes.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    recipe_data = json.load(f)

print("Dataset loaded successfully!")
print("Number of recipes:", len(recipe_data))

In [ ]:
# Display the first recipe

print(recipe_data[0])

In [ ]:
# ============================================
# Model Parameters
# ============================================

VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128

VALIDATION_SPLIT = 0.2
SEED = 42

BATCH_SIZE = 32
EPOCHS = 10

# Set to False for first training
LOAD_MODEL = False

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Parameters set successfully.")

In [ ]:
# ============================================
# Filter and Prepare Recipe Data
# ============================================

filtered_data = []

for x in recipe_data:

    if (
        "title" in x
        and x["title"] is not None
        and "directions" in x
        and x["directions"] is not None
    ):

        title = x["title"]

        directions = x["directions"]

        # Make sure directions are a list
        if isinstance(directions, list):
            directions = " ".join(directions)

        filtered_data.append(
            "Recipe for " + title + " | " + directions
        )

print("Recipes after filtering:", len(filtered_data))

In [ ]:
# ============================================
# Display an Example Recipe
# ============================================

example = filtered_data[9]

print(example)

In [ ]:
# ============================================
# Text Preprocessing
# ============================================

def pad_punctuation(s):

    s = re.sub(
        f"([{string.punctuation}])",
        r" \1 ",
        s
    )

    s = re.sub(" +", " ", s)

    return s


text_data = [
    pad_punctuation(x)
    for x in filtered_data
]

print("Text preprocessing completed.")

In [ ]:
# ============================================
# Display an Example Recipe
# ============================================

example = filtered_data[9]

print(example)

In [ ]:
# ============================================
# Text Preprocessing
# ============================================

def pad_punctuation(s):

    s = re.sub(
        f"([{string.punctuation}])",
        r" \1 ",
        s
    )

    s = re.sub(" +", " ", s)

    return s


text_data = [
    pad_punctuation(x)
    for x in filtered_data
]

print("Text preprocessing completed.")

In [ ]:
# Display processed recipe

example_data = text_data[9]

print(example_data)

In [ ]:
# ============================================
# Create TensorFlow Dataset
# ============================================

text_ds = (
    tf.data.Dataset
    .from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000, seed=SEED)
)

print("TensorFlow dataset created.")

In [ ]:
# ============================================
# Text Vectorization
# ============================================

vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1
)

print("TextVectorization layer created.")

In [ ]:
# ============================================
# Build Vocabulary
# ============================================

vectorize_layer.adapt(text_ds)

vocab = vectorize_layer.get_vocabulary()

print("Vocabulary size:", len(vocab))

In [ ]:
# ============================================
# Display Vocabulary
# ============================================

for i, word in enumerate(vocab[:20]):

    print(f"{i}: {word}")

In [ ]:
# ============================================
# Tokenize Example Recipe
# ============================================

example_tokenised = vectorize_layer(example_data)

print(example_tokenised.numpy())

In [ ]:
# ============================================
# Prepare Input and Target Sequences
# ============================================

def prepare_inputs(text):

    text = tf.expand_dims(text, -1)

    tokenized_sentences = vectorize_layer(text)

    # Input sequence
    x = tokenized_sentences[:, :-1]

    # Target sequence = next word
    y = tokenized_sentences[:, 1:]

    return x, y


train_ds = text_ds.map(
    prepare_inputs,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

print("Training dataset prepared.")

In [ ]:
# ============================================
# Inspect Training Data
# ============================================

for x_batch, y_batch in train_ds.take(1):

    print("Input shape :", x_batch.shape)
    print("Target shape:", y_batch.shape)

    print("\nFirst input sequence:")
    print(x_batch[0].numpy())

    print("\nFirst target sequence:")
    print(y_batch[0].numpy())

In [ ]:
# ============================================
# Build LSTM Model
# ============================================

inputs = layers.Input(
    shape=(None,),
    dtype="int32"
)

x = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBEDDING_DIM
)(inputs)

x = layers.LSTM(
    N_UNITS,
    return_sequences=True
)(x)

outputs = layers.Dense(
    VOCAB_SIZE,
    activation="softmax"
)(x)

lstm = models.Model(
    inputs,
    outputs
)

lstm.summary()

In [ ]:
# ============================================
# Compile Model
# ============================================

loss_fn = losses.SparseCategoricalCrossentropy()

lstm.compile(
    optimizer="adam",
    loss=loss_fn
)

print("Model compiled successfully.")

In [ ]:
# ============================================
# Text Generation Callback
# ============================================

class TextGenerator(callbacks.Callback):

    def __init__(self, index_to_word):

        super().__init__()

        self.index_to_word = index_to_word

        self.word_to_index = {
            word: index
            for index, word in enumerate(index_to_word)
        }

    def sample_from(self, probs, temperature):

        probs = np.asarray(probs).astype("float64")

        # Avoid numerical problems
        probs = np.log(probs + 1e-8) / temperature

        exp_probs = np.exp(probs - np.max(probs))

        probs = exp_probs / np.sum(exp_probs)

        return np.random.choice(
            len(probs),
            p=probs
        )

    def generate(
        self,
        start_prompt,
        max_tokens=50,
        temperature=1.0
    ):

        start_tokens = [
            self.word_to_index.get(
                x,
                1
            )
            for x in start_prompt.split()
        ]

        generated_tokens = start_tokens.copy()

        while len(generated_tokens) < max_tokens:

            x = np.array(
                [generated_tokens]
            )

            y = self.model.predict(
                x,
                verbose=0
            )

            next_token = self.sample_from(
                y[0, -1],
                temperature
            )

            generated_tokens.append(
                next_token
            )

            # Stop at unknown/padding token
            if next_token == 0:
                break

        generated_text = " ".join(
            self.index_to_word[token]
            for token in generated_tokens
            if token < len(self.index_to_word)
        )

        print("\nGenerated Recipe:")
        print("--------------------------------")
        print(generated_text)
        print("--------------------------------\n")

        return generated_text

    def on_epoch_end(self, epoch, logs=None):

        print(f"\n--- Epoch {epoch + 1} ---")

        self.generate(
            "recipe for",
            max_tokens=50,
            temperature=1.0
        )

In [ ]:
# ============================================
# Create Directories
# ============================================

os.makedirs(
    "/content/checkpoint",
    exist_ok=True
)

os.makedirs(
    "/content/models",
    exist_ok=True
)

os.makedirs(
    "/content/logs",
    exist_ok=True
)

print("Directories created.")

In [ ]:
# ============================================
# Model Checkpoint
# ============================================

model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="/content/checkpoint/lstm.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=1
)

tensorboard_callback = callbacks.TensorBoard(
    log_dir="/content/logs"
)

text_generator = TextGenerator(vocab)

print("Callbacks created.")

In [ ]:
# ============================================
# Train LSTM
# ============================================

history = lstm.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[
        model_checkpoint_callback,
        tensorboard_callback,
        text_generator
    ]
)

In [ ]:
# ============================================
# Plot Training Loss
# ============================================

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    marker="o"
)

plt.title("LSTM Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)

plt.show()

In [ ]:
# ============================================
# Save Final Model
# ============================================

MODEL_PATH = "/content/models/lstm_recipe_generator.keras"

lstm.save(MODEL_PATH)

print("Model saved successfully!")
print("Location:", MODEL_PATH)

In [ ]:
# ============================================
# Create Generator After Training
# ============================================

text_generator = TextGenerator(vocab)

# Attach trained model
text_generator.set_model(lstm)

print("Text generator ready.")

In [ ]:
# ============================================
# Generate Recipe: Roasted Vegetables
# ============================================

info = text_generator.generate(
    "recipe for roasted vegetables | chop 1",
    max_tokens=30,
    temperature=1.0
)

In [ ]:
# ============================================
# Temperature = 0.2
# ============================================

info = text_generator.generate(
    "recipe for roasted vegetables | chop 1",
    max_tokens=30,
    temperature=0.2
)

In [ ]:
# ============================================
# Generate Chocolate Ice Cream Recipe
# ============================================

info = text_generator.generate(
    "recipe for chocolate ice cream |",
    max_tokens=30,
    temperature=1.0
)

In [ ]:
# ============================================
# Temperature = 0.2
# ============================================

info = text_generator.generate(
    "recipe for chocolate ice cream |",
    max_tokens=30,
    temperature=0.2
)

In [ ]:
# ============================================
# CUSTOM RECIPE GENERATOR
# ============================================

def generate_recipe(prompt, max_tokens=50, temperature=0.8):

    print("\nPrompt:")
    print(prompt)

    print("\nGenerating...\n")

    result = text_generator.generate(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature
    )

    return result

In [ ]:
generate_recipe(
    "recipe for chicken curry |",
    max_tokens=50,
    temperature=0.8
)

In [ ]:
# ============================================
# Compare Temperature Values
# ============================================

prompt = "recipe for chocolate cake |"

print("\n\n========== TEMPERATURE 0.2 ==========")

text_generator.generate(
    prompt,
    max_tokens=30,
    temperature=0.2
)

print("\n\n========== TEMPERATURE 0.7 ==========")

text_generator.generate(
    prompt,
    max_tokens=30,
    temperature=0.7
)

print("\n\n========== TEMPERATURE 1.0 ==========")

text_generator.generate(
    prompt,
    max_tokens=30,
    temperature=1.0
)